In [1]:
from bs4 import BeautifulSoup
import json
import pandas as pd

def parse_api_documentation(html_file_path):
    with open(html_file_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')

    endpoints_data = []
    sections_data = {}

    # Find all API sections
    api_sections = soup.find_all('section', id=lambda x: x and x.startswith('api-'))

    for section in api_sections:
        section_id = section['id']
        section_name = section_id.replace('api-', '').split('-')[0] # e.g., "User" from "api-User"
        section_name = section_name.replace('/', '_')
        sections_data[section_name] = []

        # Find all article elements within the section, each representing an endpoint
        articles = section.find_all('article', class_='show-api-group') # Ensure we are within the correct group

        if not articles: # if no specific class, try general article tag within the section
            articles = section.find_all('article')

        for article in articles:
            endpoint_info = {}

            # Short description
            h1_tag = article.find('h1')
            endpoint_info['description'] = h1_tag.get_text(strip=True) if h1_tag else 'N/A'

            # HTTP Method and URL
            pre_http_tag = article.find('pre', {'data-type': ['get', 'post', 'put', 'delete', 'patch']})
            if pre_http_tag:
                endpoint_info['http_method'] = pre_http_tag['data-type'].upper()
                code_http_tag = pre_http_tag.find('code', class_='language-http')
                endpoint_info['endpoint_url'] = code_http_tag.get_text(strip=True) if code_http_tag else 'N/A'
            else:
                endpoint_info['http_method'] = 'N/A'
                endpoint_info['endpoint_url'] = 'N/A'

            # Parameters
            parameters = []
            param_table = article.find('h2', string='Параметр')
            if param_table:
                param_table_parent = param_table.find_next('table')
                if param_table_parent:
                    rows = param_table_parent.find('tbody').find_all('tr')
                    for row in rows:
                        cols = row.find_all('td')
                        if len(cols) >= 3:
                            param_name = cols[0].get_text(strip=True)
                            param_type = cols[1].get_text(strip=True)
                            param_description = cols[2].get_text(strip=True)
                            parameters.append({
                                'name': param_name,
                                'type': param_type,
                                'description': param_description
                            })
            endpoint_info['parameters'] = parameters

            # Example Response
            success_response_div = article.find('div', class_='tab-pane active', id=lambda x: x and x.startswith('success-examples-'))
            if success_response_div:
                pre_json_tag = success_response_div.find('pre', {'data-type': 'json'})
                if pre_json_tag:
                    code_json_tag = pre_json_tag.find('code', class_='language-json')
                    try:
                        # Attempt to pretty print JSON if it's valid
                        endpoint_info['example_response'] = json.dumps(json.loads(code_json_tag.get_text(strip=True)), indent=2, ensure_ascii=False)
                    except json.JSONDecodeError:
                        endpoint_info['example_response'] = code_json_tag.get_text(strip=True)
                else:
                    endpoint_info['example_response'] = 'N/A'
            else:
                endpoint_info['example_response'] = 'N/A'
            
            endpoints_data.append(endpoint_info)
            sections_data[section_name].append(endpoint_info)
    
    return endpoints_data, sections_data




In [2]:
html_file = 'c:\\temp\\UTM Rest API.html'
all_endpoints, categorized_endpoints = parse_api_documentation(html_file)
for x in all_endpoints:
    x['endpoint_url'] = x['endpoint_url'].replace('http://localhost/', '')
    #x['endpoint_url'] = x['endpoint_url'].replace('http://localhost/', '{{api_url}}')

In [24]:
# --- 1. Формирование коллекции Postman (с папками по разделам API) ---
postman_collection = {
    "info": {
        "_postman_id": "YOUR_UNIQUE_ID",
        "name": "UTM Rest API Collection",
        "schema": "https://schema.getpostman.com/json/collection/v2.1.0/collection.json"
    },
    "item": []
}

for section_name, endpoints_in_section in categorized_endpoints.items():
    # Создаем папку для каждой секции
    folder = {
        "name": section_name,
        "item": []
    }

    for ep in endpoints_in_section:
        request_item = {
            "name": ep['description'],
            "request": {
                "method": ep['http_method'],
                "header": [],
                "url": {
                    "raw": ep['endpoint_url'],
                    "host": [],
                    "path": []
                },
                "description": ep['description']
            },
            "response": [{
                "name": "Example Response",
                "originalRequest": {
                    "method": ep['http_method'],
                    "header": [],
                    "url": {
                        "raw": ep['endpoint_url'],
                        "host": [],
                        "path": []
                    }
                },
                "status": "OK",
                "code": 200,
                "_postman_previewlanguage": "json",
                "header": [
                    {
                        "key": "Content-Type",
                        "value": "application/json"
                    }
                ],
                "body": ep['example_response']
            }]
        }

        # Parse URL to separate host and path
        if ep['endpoint_url'] and '://' in ep['endpoint_url']:
            protocol, rest = ep['endpoint_url'].split('://', 1)
            if '/' in rest:
                host, path_str = rest.split('/', 1)
                request_item['request']['url']['host'] = [host]
                request_item['request']['url']['path'] = path_str.split('/')
            else:
                request_item['request']['url']['host'] = [rest]
                request_item['request']['url']['path'] = []
        
        # Add parameters to Postman request
        if ep['parameters']:
            query_params = []
            for param in ep['parameters']:
                query_params.append({
                    "key": param['name'],
                    "value": f"{{{param['name']}}}", # Use Postman variables for parameters
                    "description": param['description']
                })
            request_item['request']['url']['query'] = query_params

        folder['item'].append(request_item) # Добавляем запрос в текущую папку

    postman_collection['item'].append(folder) # Добавляем папку в коллекцию
# Сохранение коллекции Postman 
with open('c:\\temp\\UTM_postman_collection_v2.json', 'w', encoding='utf-8') as f:
     json.dump(postman_collection, f, indent=4, ensure_ascii=False)
print("Postman collection data generated.")

Postman collection data generated.


In [7]:
import os
from openai import OpenAI
import json
import requests

def generate_tool_description_local(endpoint) -> str:
    """
    Generates a description for an AI agent tool using a locally hosted LLM.
    """
    spr = f"""
        You are an AI specialized in writing concise and effective documentation for AI agent tools. 
        Your goal is to create a brief, clear, single-sentence description that a language model can use 
        to decide when to use this tool. 
        We are  creating Context model protocol server for UTM5 Rest API.
        UTM5 is ISP billing system. 
        MCP server we are create expose Ai assistant tools for query REST aPI of the this system.
        Now you are create tools description.
        Each tool have name - small descripton. List of paarameters and return expamle.
        Create description LLM friendly in maner than Ai agent will good undestend.
        The description should mention the tool's name and its primary function.
        
        Example output format: "Use the 'users/getuser' endpoint to find information about billing system."
        
        Please provide only the description text.
        """
    p = [f"{x['name']}: {x['type']}, {x['description']}" for x in endpoint['parameters']]
    upr = f"""
        For this endpoint create Description and retern it as text on Enslish
        {endpoint['description']}
        {endpoint['http_method']}
        {endpoint['endpoint_url']}
        parametrs: {p}
    """ 
    params = {
        "max_tokens": 512,
        "temperature": 0.2,
        "messages": [
        {"role": "user", "content": upr},
        {"role": "system", "content": spr}
    ]}
    response = requests.post('http://217.76.35.203:8001/v1/chat/completions', json=params).json()
    return response['choices'][0]['message']['content']


In [10]:
from time import sleep
# --- 2. Формирование файла или набора файлов по разделам API (оптимизировано для LLM) ---
base_path = 'c:\\temp\\utm5api_llmfriendly.v2\\'
for section_name, endpoints in categorized_endpoints.items():
    # Открываем файл для записи. Рекомендуется использовать расширение .md или .txt
    # для улучшения индексации и понимания содержимого LLM моделями.
    with open(f'{base_path}{section_name}_api_llm_friendly.md', 'w', encoding='utf-8') as f:
        f.write(f"# UTM5 API Section: {section_name}\n\n")
        f.write(f"This document describes the endpoints for the '{section_name}' API section for UTM5 isp billing system by Netup Co.\n\n")

        for ep in endpoints:
            try:
                sd = generate_tool_description_local(ep)
            except:
                sd = {ep['description']}
            f.write(f"## Endpoint: {ep['description']}\n")
            f.write(f"### Details:\n")
            f.write(f"- **Method:** `{ep['http_method']}`\n")
            f.write(f"- **URL:** `{ep['endpoint_url']}`\n")
            f.write(f"- **Description:** `{sd}`\n")
            f.write(f"- **Ready:** `false`\n")
            
            if ep['parameters']:
                f.write(f"### Parameters:\n")
                for param in ep['parameters']:
                    f.write(f"  - **Name:** `{param['name']}`\n")
                    f.write(f"    - **Type:** `{param['type']}`\n")
                    f.write(f"    - **Description:** {param['description']}\n")
            else:
                f.write(f"### Parameters: None\n")

            if ep['example_response'] and ep['example_response'] != 'N/A':
                f.write(f"### Example Response:\n")
                s = ep['example_response']
                start = s.find('{')
                end = s.rfind('}')
                result = s[start:end+1] if start != -1 and end != -1 else ""
                f.write(f"```\n{result}\n```\n")
            else:
                f.write(f"### Example Response: Not available\n")
            
            f.write("---\n\n") # Разделитель для ясности между эндпоинтами
            break
            sleep(2)
print("Categorized API files (LLM-friendly) data generated.")


Categorized API files (LLM-friendly) data generated.


In [14]:
block = """
## Endpoint: User - Get user data
### Details:
- **Method:** `GET`
- **URL:** `api/users`
- **desc:** `**Endpoint Description**

**Purpose**  
Retrieve a user’s profile information.

**HTTP Method**  
`GET`

**URL**  
`/api/users`

**Query Parameters**  
| Parameter | Type   | Description                                 |
|-----------|--------|---------------------------------------------|
| `user_id` | Number | The unique numeric identifier of the user.  |
| `login`   | Number | The unique numeric login of the user.       |

> **Note:**  
> • Either `user_id` or `login` (or both) can be supplied to locate the user.  
> • At least one of the parameters must be present; otherwise the request will be rejected.  

**Example Request**  
```
GET /api/users?user_id=42
```
or
```
GET /api/users?login=987654321
```

**What It Returns**  
A JSON object containing the requested user’s data (e.g., name, email, registration date, etc.).`
- **Ready:** `true`
### Parameters:
  - **Name:** `user_id`
    - **Type:** `Number`
    - **Description:** Users unique ID.
  - **Name:** `login`
    - **Type:** `Number`
    - **Description:** Users unique login.

"""

In [17]:
import re
endpoint_description_match = re.search(r'- \*\*desc:\*\* `(.*?)`', block)
endpoint_description_match#.group(1).strip()

In [3]:
categorized_endpoints.items()

dict_items([('Additonal', [{'description': 'Additonal - Delete radius session by slink_id', 'http_method': 'DELETE', 'endpoint_url': 'api/additional/radius_session', 'parameters': [], 'example_response': 'HTTP/1.1 200 OK\n  {\n    "result": "ok"\n  }'}, {'description': 'Additonal - Drop radius session', 'http_method': 'PUT', 'endpoint_url': 'api/additional/drop_radius_session', 'parameters': [], 'example_response': 'N/A'}, {'description': 'Additonal - Send PoD to radius session', 'http_method': 'PUT', 'endpoint_url': 'api/additional/disconnect_radius_session', 'parameters': [], 'example_response': 'N/A'}]), ('Customer', [{'description': 'Customer - Customer card payment', 'http_method': 'POST', 'endpoint_url': 'api/customer/card_payment', 'parameters': [], 'example_response': 'HTTP/1.1 200 OK\n{ "result" : "ok"}'}, {'description': 'Customer - Customer change account internet status', 'http_method': 'POST', 'endpoint_url': 'api/customer/change_account_int_status', 'parameters': [], 'exa

In [ ]:
from pprint import pprint
pprint(all_endpoints[0])
pprint(all_endpointsv2[0])

In [5]:
def create_description(endpoint):
    sp = """"
    You are assistant for create Context model protocol for UTM5 Rest API.
    UTM5 is ISP billing system. 
    MCP server you are create expose Ai assistant tools for query REST aPI of the this system.
    Now you are create tools description.
    Each tool have name - small descripton. List of paarameters and return expamle.
    Create description LLM friendly in maner than Ai agent will good undestend.
    """
    up = """For this endpoint create Description and retern it as text on Enslish"""
    

In [43]:
generate_tool_description_local(categorized_endpoints['User'][0])

'**Endpoint**  \n`GET /api/users`\n\n**Description**  \nRetrieves a user’s data based on the supplied query parameters.  \n- **user_id** (Number) – the unique identifier for the user.  \n- **login** (Number) – the unique login value for the user.  \n\nAt least one of these parameters must be provided; the endpoint will return the corresponding user’s information in JSON format.'

In [31]:
categorized_endpoints['User'][0]

{'description': 'User - Get user data',
 'http_method': 'GET',
 'endpoint_url': 'api/users',
 'parameters': [{'name': 'user_id',
   'type': 'Number',
   'description': 'Users unique ID.'},
  {'name': 'login', 'type': 'Number', 'description': 'Users unique login.'}],
 'example_response': 'HTTP/1.1 200 OK\n{\n  "user_id" : "1",\n  "login" : "test",\n  "password" : "0177c054",\n  "basic_account" : "1",\n  "full_name" : "",\n  "email" : "",\n  "contract_id" : "0",\n  "advance_payment" : "0",\n  "card_user" : "0",\n  "slinks" : [1],\n  "groups" : [1],\n  "accounts" : ["1"],\n  "till" : "0"\n }'}

In [ ]:
import requests


{'choices': [{'finish_reason': 'length',
   'index': 0,
   'message': {'role': 'assistant',
    'reasoning_content': 'The user says: "For this endpoint create Description and retern it as text on Enslish User - Get user data GET api/users parametrs: [\'user_id: Number, Users unique ID.\', \'login: Number, Users unique login.\']"\n\nThey want a description for the endpoint, returned as text in English. They want a description of the endpoint. They mention "User - Get user data" and GET api/users with parameters user_id and login. They want a description. So we need to produce a description of the endpoint. Possibly something like: "This endpoint retrieves user data based on user_id or login. It returns user details." They want "retern it as text on Enslish" maybe "return it as text in English". So produce a description. Let\'s produce a short description: "Endpoint: GET /api/users. Description: Retrieves user data by unique ID or login. Parameters: user_id (Number) – unique user ID; log

'**Endpoint:** `GET /api/users`\n\n**Purpose**  \nRetrieves detailed information about a user in the system. The user can be identified either by their unique numeric ID (`user_id`) or by their unique numeric login (`login`).  \n\n**Parameters**  \n- `user_id` (Number, optional) – The unique identifier assigned to the user.  \n- `login` (Number, optional) – The unique numeric login value for the user.  \n\n> *Both parameters are optional, but at least one must be supplied. If both are provided, the request will prioritize `user_id`.*\n\n**Response**  \n- **200 OK** – Returns a JSON object containing the user’s data (e.g., `id`, `login`, `name`, `email`, `created_at`, etc.).  \n- **400 Bad Request** – Missing or invalid parameters.  \n- **404 Not Found** – No user matches the supplied identifier(s).  \n\n'

In [ ]:
# Example usage:




# Сохранение коллекции Postman (я не могу создавать файлы, но вы можете скопировать этот код)
# with open('postman_collection.json', 'w', encoding='utf-8') as f:
#     json.dump(postman_collection, f, indent=4, ensure_ascii=False)
# print("Postman collection data generated.")


# --- 2. Формирование файла или набора файлов по разделам API ---
# (Я не могу создавать файлы, но вы можете скопировать этот код)
# for section_name, endpoints in categorized_endpoints.items():
#     with open(f'{section_name}_api.md', 'w', encoding='utf-8') as f:
#         f.write(f"# API Section: {section_name}\n\n")
#         for ep in endpoints:
#             f.write(f"## {ep['description']}\n")
#             f.write(f"- **Method:** {ep['http_method']}\n")
#             f.write(f"- **Endpoint:** {ep['endpoint_url']}\n")
#             if ep['parameters']:
#                 f.write(f"### Parameters:\n")
#                 for param in ep['parameters']:
#                     f.write(f"  - **{param['name']}** ({param['type']}): {param['description']}\n")
#             f.write(f"### Example Response:\n\n{ep['example_response']}\n```\n\n")
# print("Categorized API files data generated.")


# --- 3. Файл Excel ---
df_data = []
for ep in all_endpoints:
    params_str = "; ".join([f"{p['name']} ({p['type']}): {p['description']}" for p in ep['parameters']])
    df_data.append({
        "Описание": ep['description'],
        "Endpoint": ep['endpoint_url'],
        "Метод HTTP": ep['http_method'],
        "Параметры": params_str if params_str else "Нет",
        "Пример ответа": ep['example_response']
    })

df = pd.DataFrame(df_data)

# Сохранение в Excel (я не могу создавать файлы, но вы можете скопировать этот код)
# excel_output_path = 'api_documentation.xlsx'
# df.to_excel(excel_output_path, index=False)
# print(f"Excel file generated at {excel_output_path}")

print("\n--- Extracted Endpoints Data ---")
for ep in all_endpoints:
    print(f"Description: {ep['description']}")
    print(f"Method: {ep['http_method']}")
    print(f"URL: {ep['endpoint_url']}")
    print(f"Parameters: {ep['parameters']}")
    print(f"Example Response: {ep['example_response'][:200]}...") # Show first 200 chars
    print("-" * 30)

**Как использовать этот код:**

1.  **Сохраните код**: Скопируйте весь предоставленный Python-код в файл, например, `parse_api.py`.
2.  **Обновите путь к файлу**: Измените `html_file = 'c:\\temp\\UTM Rest API.htm'` на фактический путь к вашему HTML-документу.
3.  **Запустите скрипт**: Выполните его из командной строки: `python parse_api.py`.

**Объяснение кода и дальнейшие шаги:**

*   **`parse_api_documentation(html_file_path)` функция**:
    *   Открывает и читает HTML-файл.
    *   Использует `BeautifulSoup` для парсинга HTML.
    *   Ищет все `<section>` элементы, начинающиеся с `id="api-"`, чтобы определить разделы API (например, `User`).
    *   В каждом разделе ищет `<article>` элементы, которые представляют отдельные эндпоинты.
    *   Извлекает:
        *   Краткое описание (`<h1>`).
        *   HTTP-метод (`data-type` из `<pre>`).
        *   URL эндпоинта (`<code>` внутри `<pre>`).
        *   Параметры (из таблицы после `<h2>Параметр</h2>`).
        *   Пример ответа (JSON из `<pre data-type="json">`).
    *   Возвращает два набора данных: `all_endpoints` (список всех эндпоинтов) и `categorized_endpoints` (словари эндпоинтов, сгруппированных по разделам).

*   **1. Формирование коллекции Postman**:
    *   Код создает базовую структуру Postman-коллекции.
    *   Для каждого эндпоинта генерируется объект "request", включая метод, URL и описание.
    *   Параметры URL добавляются как переменные Postman (например, `{{user_id}}`) для удобства.
    *   Пример ответа добавляется в "response" объект.
    *   **Для сохранения**: Закомментированные строки с `json.dump(postman_collection, f, indent=4, ensure_ascii=False)` покажут вам, как сохранить это в JSON-файл. Вам нужно будет раскомментировать их.
    *   **Важно**: Для полноценной коллекции Postman вам может потребоваться добавить более сложную логику, например, для авторизации, переменных окружения или более детального структурирования папок в Postman.

*   **2. Формирование файлов по разделам API**:
    *   Код итерируется по `categorized_endpoints` и для каждого раздела создает отдельный Markdown-файл (например, `User_api.md`).
    *   В каждый файл записывается заголовок раздела, а затем информация о каждом эндпоинте в этом разделе, включая описание, метод, URL, параметры и пример ответа.
    *   **Для сохранения**: Закомментированные строки с `with open(f'{section_name}_api.md', 'w', encoding='utf-8') as f:` покажут вам, как сохранить это в Markdown-файлы. Вам нужно будет раскомментировать их.

*   **3. Файл Excel**:
    *   Создается DataFrame `pandas` из `all_endpoints`.
    *   Параметры объединяются в одну строку для каждой записи в Excel.
    *   **Для сохранения**: Закомментированные строки с `df.to_excel(excel_output_path, index=False)` покажут вам, как сохранить это в Excel-файл. Вам нужно будет раскомментировать их.

Дайте мне знать, если у вас возникнут вопросы по этому коду или если вы хотите, чтобы я углубился в какую-либо из частей!